# ZigNet Zig 0.16.0 — QLoRA fine-tuning

Fine-tunes a PEFT adapter for `fulgidus/zignet-qwen2.5-coder-7b` using JSONL in `data/training/v0.16.0/`. Runtime: Google Colab GPU (T4/A100 preferred). This notebook refuses to silently fall back to the old dataset. Confirm the Hugging Face model is a Transformers checkpoint and inspect its license/revision before running.

In [ ]:
# 1. Check runtime and set paths
import os, json, glob, platform, subprocess, sys
from pathlib import Path

ROOT = Path('/content/zignet-latest')
DATA_DIR = ROOT / 'data' / 'training' / 'v0.16.0'
OUTPUT_DIR = Path('/content/zignet-016-qlora-output')
MODEL_ID = 'fulgidus/zignet-qwen2.5-coder-7b'
MAX_SEQ_LENGTH = 1024
assert 'COLAB_GPU' in os.environ or Path('/content').exists(), 'Run this notebook in Google Colab or adapt paths for a local environment.'
print('Python:', sys.version)
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as e:
    print('GPU unavailable:', e)


In [ ]:
# 2. Clone/pull repository and build deterministic train/validation/test splits from category JSONL files
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/suissa/zignet-latest.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)

DATA_DIR.mkdir(parents=True, exist_ok=True)
split_names = ('dataset-train.jsonl', 'dataset-validation.jsonl', 'dataset-test.jsonl')
source_files = sorted(p for p in DATA_DIR.glob('*.jsonl') if p.name not in split_names)
print('Dataset directory:', DATA_DIR)
print('Source JSONL files:', len(source_files))
if not source_files:
    raise FileNotFoundError(f'No category JSONL files found in {DATA_DIR}; refusing to use legacy data.')

records = []
for source in source_files:
    with source.open(encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f'Invalid JSON in {source.name}:{line_no}: {e}') from e
            missing = {'instruction', 'input', 'output', 'metadata'} - set(row)
            if missing:
                raise ValueError(f'{source.name}:{line_no} missing fields: {sorted(missing)}')
            row['_source_file'] = source.name
            records.append(row)

if len(records) < 30:
    raise ValueError(f'Only {len(records)} examples found; refusing to create tiny splits.')

# Split by scenario family to reduce near-duplicate leakage across splits.
families = {}
for row in records:
    meta = row.get('metadata') or {}
    family = meta.get('split_group') or meta.get('scenario_family') or row['_source_file']
    families.setdefault(str(family), []).append(row)
family_names = sorted(families)
if len(family_names) < 10:
    raise ValueError(f'Only {len(family_names)} independent split groups; need at least 10.')

train_rows, val_rows, test_rows = [], [], []
for i, family in enumerate(family_names):
    bucket = i % 10
    target = test_rows if bucket == 0 else val_rows if bucket == 1 else train_rows
    target.extend(families[family])

for rows in (train_rows, val_rows, test_rows):
    for row in rows:
        row.pop('_source_file', None)

train_path = DATA_DIR / 'dataset-train.jsonl'
val_path = DATA_DIR / 'dataset-validation.jsonl'
test_path = DATA_DIR / 'dataset-test.jsonl'
for path, rows in ((train_path, train_rows), (val_path, val_rows), (test_path, test_rows)):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

validated = sum(bool((r.get('metadata') or {}).get('compiler_validated')) for r in records)
print(f'Wrote splits: train={len(train_rows)}, validation={len(val_rows)}, test={len(test_rows)}')
print(f'Compiler-validated source examples: {validated}/{len(records)}')
if validated == 0:
    print('BLOCKED FOR TRAINING: current repository seeds are snippet_only/unvalidated. Run Zig 0.16 compiler validation and review first; Cell 4 will refuse to train this dataset.')


In [ ]:
# 3. Install libraries (pin pyarrow to remain compatible with Colab's preinstalled cuDF)
%pip -q install -U "pyarrow>=19,<24" transformers datasets accelerate peft trl bitsandbytes safetensors huggingface_hub

# If this cell upgrades core packages, restart the runtime before continuing.


In [ ]:
# 4. Load JSONL and validate schema — refuse to fine-tune unvalidated seed data
from datasets import load_dataset
data_files = {'train': str(train_path), 'validation': str(val_path), 'test': str(test_path)}
raw = load_dataset('json', data_files=data_files)
required = {'instruction', 'input', 'output', 'metadata'}
for split in raw:
    missing = required - set(raw[split].column_names)
    if missing:
        raise ValueError(f'{split}: missing required fields {missing}; columns={raw[split].column_names}')
    print(split, len(raw[split]), raw[split].column_names)
if len(raw['train']) < 20:
    raise ValueError('Too few training examples. Generate a larger dataset before training.')

def is_compiler_validated(example):
    meta = example.get('metadata') or {}
    return bool(meta.get('compiler_validated')) and meta.get('validation_status') in {'compiler_validated', 'passed'}

for split in ('train', 'validation', 'test'):
    validated_count = sum(is_compiler_validated(row) for row in raw[split])
    print(f'{split}: compiler-validated {validated_count}/{len(raw[split])}')
    if validated_count != len(raw[split]):
        raise ValueError(f'{split} contains unvalidated examples. Do not fine-tune on seed/snippet-only data; validate every example with Zig 0.16.0 and review the migration semantics first.')


In [ ]:
# 5. Load model in 4-bit and configure LoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=False)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print('Model loaded. Candidate linear modules:', [n for n, m in model.named_modules() if n.endswith(('q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'))][:30])


In [ ]:
# 6. Select LoRA targets and format instruction data
# Verify these module names from the previous cell for this checkpoint; adjust if architecture differs.
candidate_suffixes = {'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'}
available = {n.split('.')[-1] for n, m in model.named_modules() if n.split('.')[-1] in candidate_suffixes and hasattr(m, 'weight')}
target_modules = sorted(available)
if not target_modules:
    raise RuntimeError('No expected Qwen projection modules found; inspect architecture and set target_modules explicitly.')
print('LoRA target modules:', target_modules)
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=target_modules)

def format_example(example):
    instruction = example['instruction'].strip()
    inp = example['input'].strip()
    out = example['output'].strip()
    user_text = instruction + ('\n\nInput:\n' + inp if inp else '')
    messages = [{'role':'user','content':user_text}, {'role':'assistant','content':out}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = f'### Instruction\n{instruction}\n\n### Input\n{inp}\n\n### Response\n{out}{tokenizer.eos_token}'
    return {'text': text}

formatted = raw.map(format_example, remove_columns=raw['train'].column_names)
print(formatted['train'][0]['text'][:1000])


In [ ]:
# 7. Configure training — conservative settings for a 12 GB GPU
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    gradient_checkpointing=True,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim='paged_adamw_8bit',
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,
    max_grad_norm=0.3,
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=formatted['train'],
    eval_dataset=formatted['validation'],
    peft_config=lora_config,
    processing_class=tokenizer,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
)
print('Trainer ready. For an initial smoke test, use a small subset and max_steps=10 before a full run.')


In [ ]:
# 8. Train
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train()
print(train_result.metrics)
print('Peak allocated VRAM GiB:', round(torch.cuda.max_memory_allocated() / 1024**3, 2))
print('Peak reserved VRAM GiB:', round(torch.cuda.max_memory_reserved() / 1024**3, 2))


In [ ]:
# 9. Evaluate held-out validation and test sets
print('Validation:', trainer.evaluate(formatted['validation']))
test_metrics = trainer.evaluate(formatted['test'], metric_key_prefix='test')
print('Test:', test_metrics)


In [ ]:
# 10. Save adapter, tokenizer and run metadata
adapter_dir = OUTPUT_DIR / 'adapter'
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
metadata = {
    'base_model': MODEL_ID,
    'dataset_dir': str(DATA_DIR),
    'dataset_counts': {k: len(v) for k, v in raw.items()},
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora': {'r': 8, 'alpha': 16, 'dropout': 0.05, 'target_modules': target_modules},
    'compute_dtype': str(compute_dtype),
    'train_metrics': train_result.metrics,
    'test_metrics': test_metrics,
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
}
(OUTPUT_DIR / 'run-metadata.json').write_text(json.dumps(metadata, indent=2, default=str))
print('Saved:', adapter_dir)


## Important notes

- This trains a PEFT adapter; it does not merge weights or produce a GGUF file.
- The exact checkpoint may have a different architecture or chat template. Inspect the model card and module list before training.
- For RTX 4070 12 GB, start with `MAX_SEQ_LENGTH=512` and a short pilot if OOM occurs.
- A 1 vCPU / 1 GB RAM VPS cannot fine-tune a 7B model. Use it only for streaming validation, deduplication and dataset shard generation.
- Compiler validation belongs in the data-generation pipeline: this notebook trains from JSONL and does not prove the Zig examples compile.